# 2-Stage YOLOv11-seg Training for Zero-Shot Vape Grasping (Dataset A)

This Google Colab notebook trains our real-time instance segmentation model (`yolo11s-seg`) for tabletop robotic grasping:
1. **Environment & GPU Check**: Installs `ultralytics>=8.3.0` and verifies T4 / L4 / A100 availability.
2. **Dataset A Unpack, SAM 2 BBox-to-Polygon Upgrade & Hygiene Validation**: Upgrades public 4-value bounding-box annotations into tight instance masks via `upgrade_bbox_labels_with_sam` and verifies zero cross-split SHA-256 leakage and strict isolation of **Unseen Vape E**.
3. **Stage 1 — General Vape Pre-Finetuning**: Trains `yolo11s-seg.pt` on ~3,000–5,000 curated public vape + hard-negative images with full rotational (`degrees=180.0`) and hue (`hsv_h=0.4`) invariance.
4. **Stage 2 — Overhead Tabletop Adaptation**: Fine-tunes Stage 1 `best.pt` for 25 epochs at `lr0=5e-4` with `AdamW` (`freeze=10`, `patience=15`) on ~200–250 local overhead photos (Vapes A–D, AirPods/pens/USB hard negatives, and zero-vape background scenes).
5. **Zero-Shot Unseen Vape E Evaluation & Checkpoint Export**: Evaluates mask mAP on the held-out `test` split (Unseen Vape E) and downloads `best.pt` for local Mac deployment.

In [ ]:
# Cell 1: Install Ultralytics & verify Colab GPU (T4 / L4 / A100)
!nvidia-smi
!pip install -q "ultralytics>=8.3.0"

import sys
from pathlib import Path

import torch
import ultralytics

ultralytics.checks()
assert torch.cuda.is_available(), (
    "Enable a GPU runtime in Colab: Runtime -> Change runtime type -> T4 / L4 / A100 GPU"
)
print(f"Using GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Mount Google Drive, unpack Dataset A, upgrade BBoxes -> SAM 2, & validate splits
import shutil
import zipfile

from google.colab import drive

# Class-schema configuration (must match --num-classes used in sam_masker):
# - 1-class mode: CLASS_NAMES = ["vape"] (nc=1; lookalikes such as AirPods/pens
#   are left unlabeled as negative background).
# - 2-class mode: CLASS_NAMES = ["vape", "non_vape"] (nc=2; lookalikes explicitly
#   annotated as class ID 1, e.g. with `--num-classes 2` in sam_masker).
CLASS_NAMES = ["vape"]
DRIVE_ROOT = Path("/content/drive/MyDrive/leukerbad_robotics")
WORK_DIR = Path("/content/datasets")
PUBLIC_ROOT = WORK_DIR / "dataset_a_public"
OVERHEAD_ROOT = WORK_DIR / "dataset_a_overhead"

drive.mount("/content/drive")

REPO_SRC = DRIVE_ROOT / "leukerbad-object-detection-robotics" / "src"
if REPO_SRC.exists() and str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

from leukerbad.dataset.sam_masker import upgrade_bbox_labels_with_sam
from leukerbad.dataset.validate import (
    validate_yolo_seg_dataset,
    write_yolo_dataset_yaml,
)

for zip_name, target_dir in [
    ("dataset_a_public.zip", PUBLIC_ROOT),
    ("dataset_a_overhead.zip", OVERHEAD_ROOT),
]:
    zip_path = DRIVE_ROOT / zip_name
    if zip_path.exists() and not target_dir.exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(target_dir)
        print(f"Extracted {zip_path} -> {target_dir}")

# Upgrade any 4-number public YOLO bounding boxes into tight SAM 2 instance polygons
for split in ("train", "val", "test"):
    img_dir = PUBLIC_ROOT / "images" / split
    lbl_dir = PUBLIC_ROOT / "labels" / split
    if img_dir.exists() and lbl_dir.exists():
        stats = upgrade_bbox_labels_with_sam(
            images_dir=img_dir,
            labels_dir=lbl_dir,
            output_labels_dir=lbl_dir,
            sam_weights="sam2.1_b.pt",
        )
        print(f"[Public {split}] SAM upgrade stats: {stats}")

public_yaml = write_yolo_dataset_yaml(PUBLIC_ROOT, CLASS_NAMES)
overhead_yaml = write_yolo_dataset_yaml(OVERHEAD_ROOT, CLASS_NAMES)

for label, ds_root in [("Public Stage-1", PUBLIC_ROOT), ("Overhead Stage-2", OVERHEAD_ROOT)]:
    report = validate_yolo_seg_dataset(ds_root, num_classes=len(CLASS_NAMES))
    print(
        f"{label} validation: valid={report.is_valid}, splits={report.split_counts}, "
        f"polygons={report.polygon_count}, near_duplicates={len(report.near_duplicates)}"
    )
    if report.near_duplicates:
        print(f"{label} cross-split near-duplicates: {report.near_duplicates[:5]}")
    assert report.is_valid, f"{label} failed dataset hygiene checks: {report.errors[:5]}"

In [ ]:
# Cell 3: Stage 1 — General Vape Pre-Finetuning (3,000–5,000 curated public images)
from ultralytics import YOLO

stage1_model = YOLO("yolo11s-seg.pt")
stage1_results = stage1_model.train(
    data=str(public_yaml),
    epochs=60,
    imgsz=640,
    batch=32,
    device=0,
    seed=0,
    degrees=180.0,   # Full 360-degree tabletop rotation invariance
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.4,       # Strong color/hue randomization so geometry beats brand color
    hsv_s=0.6,
    hsv_v=0.4,
    copy_paste=0.2,  # Multi-instance clutter & occlusion synthesis
    project="/content/runs/vape_seg",
    name="stage1_public_pretrain",
    exist_ok=True,
)

stage1_best = (
    Path(stage1_model.trainer.best)
    if getattr(stage1_model, "trainer", None) is not None
    and getattr(stage1_model.trainer, "best", None)
    else Path(stage1_results.save_dir) / "weights" / "best.pt"
)
assert stage1_best.exists(), f"Stage 1 best checkpoint missing at {stage1_best}"
print(f"Stage 1 complete: {stage1_best}")

In [ ]:
# Cell 4: Stage 2 — Overhead Tabletop Adaptation (Vapes A–D + hard negatives + empty bg)
# Explicit optimizer="AdamW" prevents Ultralytics optimizer="auto" from silently
# overriding lr0; freeze=10 + patience=15 prevents overfitting on ~200–250 local images.
stage2_model = YOLO(str(stage1_best))
stage2_results = stage2_model.train(
    data=str(overhead_yaml),
    epochs=25,
    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.05,
    patience=15,
    freeze=10,
    seed=0,
    imgsz=640,
    batch=16,
    device=0,
    degrees=180.0,
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.4,
    copy_paste=0.2,
    mosaic=0.5,
    close_mosaic=10,
    project="/content/runs/vape_seg",
    name="stage2_overhead_tabletop",
    exist_ok=True,
)

stage2_best = (
    Path(stage2_model.trainer.best)
    if getattr(stage2_model, "trainer", None) is not None
    and getattr(stage2_model.trainer, "best", None)
    else Path(stage2_results.save_dir) / "weights" / "best.pt"
)
assert stage2_best.exists(), f"Stage 2 best checkpoint missing at {stage2_best}"
print(f"Stage 2 complete: {stage2_best}")

In [ ]:
# Cell 5: Zero-shot evaluation on held-out Unseen Vape E test split + export best.pt for Mac
from google.colab import files

final_model = YOLO(str(stage2_best))
unseen_metrics = final_model.val(
    data=str(overhead_yaml),
    split="test",  # Held-out Unseen Vape E + unseen hard negatives
    imgsz=640,
    device=0,
)

print("=== Zero-Shot Held-Out Unseen Vape E Metrics ===")
print(f"Box  mAP50:    {unseen_metrics.box.map50:.4f} | mAP50-95: {unseen_metrics.box.map:.4f}")
print(f"Mask mAP50:    {unseen_metrics.seg.map50:.4f} | mAP50-95: {unseen_metrics.seg.map:.4f}")

export_dst = DRIVE_ROOT / "vape_yolo11s_seg_best.pt"
export_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(stage2_best, export_dst)
print(f"Backed up checkpoint to Google Drive: {export_dst}")

files.download(str(stage2_best))